# 04 · Toy fields: Gaussian random fields and their gas cells

A toy field is a gas whose statistics we choose: a Gaussian random field with a known spectrum, isotropic or with the
anisotropy and the acoustic feature of a real survey. It enters the optical depth the same way as a simulation does:
as **gas cells**. The gas is a set of Voronoi cells, in which every point of space belongs to the cell of its nearest
generating point, and inside a cell the gas is uniform. `Cells` holds the generating points, their periodic box and the
cell volumes, and, per absorbing species, what the optical depth needs (review notes, Sec. GasProperties):

| field | unit | required |
|---|---|---|
| `number_density` | comoving cm$^{-3}$ | yes |
| `temperature` | K | yes |
| `velocity` | km/s (peculiar), one vector per cell | yes |
| `turbulence` | m/s ($b_{\rm turb}$) | no (zero) |
| `collisional` | GHz ($A_{\rm coll}$) | no (zero) |

This notebook builds the toy fields of SkewerJAX (`skewerjax.simulations`, a source of cells like a snapshot): a
Gaussian random field (`PowerLawGRF`) and its statistics, isotropic and anisotropic with a BAO feature (§1), and the
cells it defines, sampled on the grid or at simulation-like generating points (§2). Notebook 05 builds the same
container from a simulation snapshot. Runtime: ~3 min.

**Requires:** nothing: a Gaussian random field and its cells.  
**Independent of:** lines, cosmology, skewers.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)

In [ ]:
plot_dag(('samplers', 'fields',)); plt.show()

## 1 · Gaussian random fields: `PowerLawGRF`

`PowerLawGRF` draws a periodic Gaussian random field in 1, 2 or 3 dimensions, with spectrum

$$
P(k) = A_s \left(\frac{k}{k_0}\right)^{n_s - 1} .
$$

The grid is given either by $(L, N)$ or by two of $(k_{\min}, k_{\max}, N_k)$. The normalisation is that of the P1D
estimator of the review notes, $\langle \lvert\delta_k\rvert^2\rangle / V = P(k)$, so `variance()` returns the
expected $\sigma^2$. `sample(key)` is jitted, and `jax.vmap` over keys draws a batch. Optional terms add an anisotropy and a BAO
feature (below).

In [ ]:
from skewerjax.simulations import PowerLawGRF
key = jax.random.PRNGKey(1)
g1 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=-0.5, L=100.0, N=1024)
g2 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=-1.5, L=(50.0, 100.0), N=(128, 256))     # anisotropic box: y = 2 x
g3 = PowerLawGRF(A_s=4.0, k_0=2.0, n_s=-1.25, k_min=(2 * np.pi / 50,) * 3, N_k=(256,) * 3)   # mode (b): N_k modes per axis
for g in (g1, g2, g3):
    print(g, f"sigma = {float(jnp.sqrt(g.variance())):.3f}")
d1, d2, d3 = g1.sample(key), g2.sample(key), g3.sample(key)
iz = d3.shape[2] // 2                                                              # the slice z = L/2
fig, axs = plt.subplots(1, 3, figsize=(24, 7.5), layout="constrained", gridspec_kw={"width_ratios": [1.3, 0.75, 1]})
axs[0].plot(g1.x_grid()[0], d1, color=EC["density"], lw=1); axs[0].set(xlabel="x [Mpc]", ylabel=r"$\delta$", title="1D, $n_s = -0.5$")
Lx, Ly = g2.L
im = axs[1].imshow(np.asarray(d2).T, origin="lower", extent=(0, Lx, 0, Ly), cmap=CMAPS["diverging"], aspect="equal")
axs[1].set(xlabel="x [Mpc]", ylabel="y [Mpc]", title="2D, box 50 × 100 Mpc")
L3 = g3.L[0]
axs[2].imshow(np.asarray(d3[:, :, iz]).T, origin="lower", extent=(0, L3, 0, L3), cmap=CMAPS["diverging"], aspect="equal")
axs[2].set(xlabel="x [Mpc]", ylabel="y [Mpc]", title=f"3D, {d3.shape[0]}³ cells, slice z = L/2 (index {iz})")
add_panel_letters(axs); plt.show()

**Isotropic statistics.** The measured power spectrum of many draws recovers the input in every dimension.
The estimator is the one that defines the normalisation:

$$
\hat P(k) = \frac{\lvert\delta_k\rvert^2}{V},
$$

where $\delta_k$ is the discrete Fourier transform times the cell volume. It is averaged over the modes of each bin of
$\lvert k\rvert$ and over the draws, and the error bars are the scatter between draws divided by the square root of
their number.

The real-space correlation function is the same information in configuration space,

$$
\hat\xi(\mathbf r) = \frac{1}{N}\sum_{\mathbf x} \delta(\mathbf x)\, \delta(\mathbf x + \mathbf r),
\qquad
\xi_{\rm in}(\mathbf r) = \frac{1}{V}\sum_{\mathbf k} P(k)\, e^{i\mathbf k\cdot\mathbf r} .
$$

- $\hat\xi$ is computed from the same transform (Wiener–Khinchin) and averaged over the grid separations of each bin
  of $\lvert r\rvert$, up to a quarter of the box.
- $\xi_{\rm in}$ is the inverse transform of the input $P(k)$ on the same grid, with the box's own modes: the exact
  correlation of the discrete periodic field, so that $\langle\hat\xi\rangle = \xi_{\rm in}$. It has no $k = 0$ mode,
  so it integrates to zero over the box and turns negative at large separations.
- At zero separation it is the variance, $\xi_{\rm in}(0) = \sigma^2$ (`variance()`).
- The error of $\hat\xi$ is its exact Gaussian covariance over the draws. The modulus squared of every mode is
  exponentially distributed, with variance $P(k)^2$, so

$$
\mathrm{Cov}\big(\hat\xi_b, \hat\xi_{b'}\big) = \frac{2}{V^2 N_{\rm draws}} \sum_{\mathbf k \neq 0} P(k)^2\, c_b(\mathbf k)\, c_{b'}(\mathbf k),
\qquad c_b(\mathbf k) = \big\langle \cos(\mathbf k\cdot\mathbf r) \big\rangle_{\mathbf r \in b} .
$$

The figure shows, per dimension: $P(\lvert k\rvert)$ and the input; the ratio minus one; $\xi(\lvert r\rvert)$ and
$\xi_{\rm in}$; their difference in units of its error.

In [ ]:
from skewerjax.utils.plotting import plot_isotropic_recovery, plot_kpar_kperp_maps, plot_pk_mu_wedges, MU_WEDGES
draws = lambda g, n, seed: np.asarray(jax.vmap(g.sample)(jax.random.split(jax.random.PRNGKey(seed), n)))
cases = {"1D, 200 draws": (g1, draws(g1, 200, 2)), "2D (50 × 100 Mpc), 40 draws": (g2, draws(g2, 40, 3)),
         "3D (512³), 4 draws": (g3, draws(g3, 4, 4))}
fig, axs, rec = plot_isotropic_recovery(cases)
add_panel_letters(axs.T.ravel())
fig.suptitle(r"PowerLawGRF, isotropic statistics: $P(|k|)$ and $\xi(|r|)$ recover the input in 1D, 2D and 3D", fontsize=22)
plt.show()
for name, r in rec.items():
    print(f"{name:28s}: P(|k|) chi2/dof = {r['chi2_dof']:.2f}, max |P / P_in - 1| = {r['max_rel']:.3f}; xi(|r|): mean pull {r['xi_mean_pull']:+.2f}, "
          f"leading mode {r['xi_lead']:.0%} of the variance, chi2/dof with the covariance {r['xi_chi2_cov']:.2f} ({r['xi_dof']} dof)")

*Reading the figure.* The $\xi$ points of one panel are not independent. Every bin is set mostly by the same few
largest-scale modes of the box, whose realised power scatters by $1/\sqrt{n}$ for $n$ modes, so the points move up or
down together: the title of each residual panel gives the share of the variance in that common mode, and the
$\chi^2$ that uses the full covariance. A shift of every point by about one standard deviation is one fluctuation of
those modes, not a bias: its sign changes with the seed, while $P(k)$, whose bins are independent, shows none.

**Anisotropic decompositions.** A skewer survey singles out one direction, the line of sight. Here it is the
last axis of the grid: $k_\parallel$ is the wavenumber along it and $k_\perp$ the one across it. The isotropic average
above is justified only if the measured power depends on $\lvert k\rvert$ alone. Two views test this in 2D and 3D.

- **The $(k_\parallel, k_\perp)$ plane.** The measured power is the colour, with the measured iso-power contours
  (solid) drawn on the input ones (dashed) at the same levels.
  - The $\chi^2$ printed below the figure is that of $(P_{\rm meas} - P_{\rm in})/\sigma_P$ per bin. Here
    $\sigma_P = P_{\rm in}/\sqrt{n\,N_{\rm draws}}$ comes from mode counting: $\lvert\delta_k\rvert^2$ of a Gaussian field
    is exponentially distributed, so the mean of $n$ independent modes has a relative error $1/\sqrt n$. $P_{\rm in}$ is
    averaged over the modes of each bin.
  - Both axes are logarithmic (one mode per bin at low $k$, logarithmic bins above), so the modes with
    $k_\parallel = 0$ or $k_\perp = 0$ are not shown.
  - The anisotropic 2D box ($L_y = 2 L_x$) has different fundamental modes along the two directions, $2\pi/100$ and
    $2\pi/50\ \mathrm{Mpc^{-1}}$, and it still follows one $P(k)$.
- **Wedges in $\mu = k_\parallel/\lvert k\rvert$.** $\mathrm{Ly}\alpha$ 3D analyses show their measurements in
  wedges of $\mu$, the cosine of the angle to the line of sight, from far from it ($0 \le \mu < 0.5$) to along it
  ($0.95 \le \mu \le 1$), with the model on the data (Gordon et al. 2023,
  [arXiv:2308.10950](https://arxiv.org/abs/2308.10950), Fig. 4, for the correlation function in $\mu = r_\parallel/r$).
  Here the same four wedges hold the power spectrum: one panel per case with every wedge in its own colour (points:
  measured; dashed: the input averaged over the modes of each $(k, \mu)$ bin), and below it the ratio minus one in the
  same colours. For an isotropic input every wedge must recover $P(k)$; a redshift-space distortion would separate
  them.

Full checks: `scripts/validation/check_summary_statistics.py`.

In [ ]:
aniso = {name: c for name, c in cases.items() if c[0].ndim > 1}          # the 2D and 3D draws
fig, axs, chi2_map = plot_kpar_kperp_maps(aniso)
add_panel_letters(axs)
fig.suptitle(r"The $(k_\parallel, k_\perp)$ plane: measured contours on the input ones", fontsize=21)
plt.show()
print({name: f"map chi2/dof = {x:.2f}" for name, x in chi2_map.items()})

In [ ]:
fig, axs, chi2_mu = plot_pk_mu_wedges(aniso)
add_panel_letters(axs.T.ravel())
fig.suptitle(r"$P(k, \mu)$ in four wedges of $\mu = k_\parallel/|k|$, one colour per wedge: every wedge of an isotropic field recovers $P(k)$",
             fontsize=21)
plt.show()
for name, x in chi2_mu.items():
    print(f"{name:28s}: chi2/dof per wedge", {f"[{a:g}, {b:g})": round(c, 2) for a, b, c in zip(MU_WEDGES[:-1], MU_WEDGES[1:], x)})

**Anisotropy and a BAO feature.** The $\mathrm{Ly}\alpha$ forest is anisotropic, because peculiar velocities move
the absorbers along the line of sight, and it carries the baryon acoustic feature. `PowerLawGRF` can add both, as
optional terms (all off by default, which leaves the isotropic draws above bit for bit unchanged):

$$
P(k, \mu) = \big[P_{\rm pl}(k) + P_{\rm BAO}(k)\big]\,\big(1 + \beta\mu^2\big)^2\, D(k\mu\,\sigma_{\rm FoG}),
\qquad D(x) = \frac{1}{1 + x^2}\ \ \text{or}\ \ e^{-x^2},
$$

- $\mu = \lvert k_\parallel\rvert/\lvert k\rvert$ is the cosine of the angle to the line of sight, a grid axis (`los_axis`,
  by default the last one, the $k_\parallel$ of the maps and wedges); in 1D, $\mu = 1$.
- $(1 + \beta\mu^2)^2$ is a Kaiser-like boost along the line of sight, and $D$ the Fingers-of-God damping of one
  velocity dispersion $\sigma_{\rm FoG}$ (Lorentzian by default, or Gaussian).
- The BAO feature is a Gaussian bump in the correlation function,
  $\xi_{\rm BAO}(r) = A_{\rm BAO}\big[g(r - r_{\rm BAO}) + g(r + r_{\rm BAO})\big]$ with
  $g(u) = e^{-u^2/2\sigma_{\rm BAO}^2}$, added to $P$ through its exact Fourier transform in each dimension. With
  $G(k) = \sigma_{\rm BAO}\sqrt{2\pi}\, e^{-k^2\sigma_{\rm BAO}^2/2}$:

$$
P_{\rm BAO}^{\rm 1D} = 2A_{\rm BAO}\, G \cos(k r_{\rm BAO}),
\qquad
P_{\rm BAO}^{\rm 3D} = 4\pi A_{\rm BAO}\, G\left[\sigma_{\rm BAO}^2 \cos(k r_{\rm BAO}) + \frac{r_{\rm BAO}\sin(k r_{\rm BAO})}{k}\right],
$$

  and in 2D the Hankel transform $2\pi\int_0^\infty r\,\xi_{\rm BAO}(r)\, J_0(kr)\, dr$, by quadrature.
- A Gaussian field needs $P \ge 0$ at every mode. The two factors are never negative, so `PowerLawGRF` checks
  $P_{\rm pl} + P_{\rm BAO} \ge 0$ at construction and raises an error naming the $k$ where it fails: a large bump
  makes the oscillation of $P_{\rm BAO}$ deeper than the power law.

Every expectation follows the same terms on the box's own modes: the input $P(k, \mu)$ of every mode in the ratios,
the maps and the wedges, its inverse transform for $\xi_{\rm in}$, and its average over directions for the smooth curve
of $P(\lvert k\rvert)$.

The configuration below has $\beta = 1$ and $\sigma_{\rm FoG} = 1$ Mpc in 1D, 2D and 3D. The base spectra are bluer
than above ($n_s = 0.5$ in 1D and 2D, $-0.25$ in 3D), with less large-scale power, so that the bump stands out in
$\xi$; it sits at $r_{\rm BAO} = 16$ Mpc in 1D and 8 Mpc in 2D and 3D, inside the quarter of the box where $\xi$ is
shown, with $A_{\rm BAO}$ at about two thirds of the largest value that keeps $P \ge 0$. The steps and the figures are
those of the isotropic case.

In [ ]:
rsd = dict(beta=1.0, sigma_fog=1.0)                                                 # Kaiser and Fingers of God, along the last axis
b1 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=0.5, L=100.0, N=1024, A_bao=0.15, r_bao=16.0, sigma_bao=1.6, **rsd)
b2 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=0.5, L=(50.0, 100.0), N=(128, 256), A_bao=0.025, r_bao=8.0, sigma_bao=0.8, **rsd)
b3 = PowerLawGRF(A_s=4.0, k_0=2.0, n_s=-0.25, k_min=(2 * np.pi / 50,) * 3, N_k=(128,) * 3, A_bao=0.04, r_bao=8.0, sigma_bao=0.8, **rsd)
for g in (b1, b2, b3):
    print(g, f"sigma = {float(jnp.sqrt(g.variance())):.3f}")
try:
    PowerLawGRF(A_s=1.0, k_0=1.0, n_s=0.5, L=100.0, N=1024, A_bao=1.0, r_bao=16.0, sigma_bao=1.6)
except ValueError as e:
    print("a bump too large:", e)
cases_b = {"1D, 200 draws": (b1, draws(b1, 200, 12)), "2D (50 × 100 Mpc), 40 draws": (b2, draws(b2, 40, 13)),
           "3D (256³), 8 draws": (b3, draws(b3, 8, 14))}
fig, axs, rec_b = plot_isotropic_recovery(cases_b)
add_panel_letters(axs.T.ravel())
fig.suptitle(r"Anisotropic PowerLawGRF with a BAO bump: $P(|k|)$ and $\xi(|r|)$ recover the input in 1D, 2D and 3D", fontsize=22)
plt.show()
for name, r in rec_b.items():
    print(f"{name:28s}: P(|k|) chi2/dof = {r['chi2_dof']:.2f}, max |P / P_in - 1| = {r['max_rel']:.3f}; xi(|r|): mean pull {r['xi_mean_pull']:+.2f}, "
          f"leading mode {r['xi_lead']:.0%} of the variance, chi2/dof with the covariance {r['xi_chi2_cov']:.2f} ({r['xi_dof']} dof)")

In [ ]:
aniso_b = {name: c for name, c in cases_b.items() if c[0].ndim > 1}
fig, axs, chi2_map_b = plot_kpar_kperp_maps(aniso_b)
add_panel_letters(axs)
fig.suptitle(r"The $(k_\parallel, k_\perp)$ plane of the anisotropic input: measured contours on the input ones", fontsize=21)
plt.show()
print({name: f"map chi2/dof = {x:.2f}" for name, x in chi2_map_b.items()})

In [ ]:
fig, axs, chi2_mu_b = plot_pk_mu_wedges(aniso_b)
add_panel_letters(axs.T.ravel())
fig.suptitle(r"$P(k, \mu)$ in four wedges of $\mu$: the anisotropic input separates them, and every wedge recovers its own input", fontsize=21)
plt.show()
for name, x in chi2_mu_b.items():
    print(f"{name:28s}: chi2/dof per wedge", {f"[{a:g}, {b:g})": round(c, 2) for a, b, c in zip(MU_WEDGES[:-1], MU_WEDGES[1:], x)})

*Reading the figures.*

- **The BAO feature.** $\xi(\lvert r\rvert)$ has its bump at $r_{\rm BAO}$, measured and input alike, and $P(\lvert k\rvert)$
  oscillates with period $2\pi/r_{\rm BAO}$ in $k$, damped above $k \sim 1/\sigma_{\rm BAO}$. The ratios stay flat:
  the measurement follows the wiggles mode by mode.
- **The anisotropy.** The iso-power contours of the maps are squashed: at fixed $\lvert k\rvert$ the power is larger along
  the line of sight on large scales (Kaiser) and smaller on small scales (Fingers of God). The wedges separate in the
  same way, by up to $(1 + \beta)^2 = 4$ at low $k$, and cross where the damping takes over; each one recovers its
  own input.
- The smooth curve of $P(\lvert k\rvert)$ is the input averaged over directions; the measurement follows the average over
  the modes of each bin, which the ratio uses.

## 2 · From the field to gas cells

A toy field gives full control over the statistics of the gas, and `grf_cells` turns it into the same container as a
simulation. How the cells sample the field is then a choice we can vary on purpose: sampling it in a simulation-like
way gives a controlled benchmark of how that sampling changes the measured statistics.
`grf_cells(delta, grf, n_bar, generators=...)` sets $n = \bar n\,(1 + \delta)$ in every cell, with no transform or clipping ($\bar n$ is the mean
density, because $\delta$ has no $k = 0$ mode; the amplitude must keep $1 + \delta > 0$, and a warning is issued
otherwise). Three choices of generating points, as many as grid cells in each:

- **(a) grid cells** (default): the generating points are the grid nodes $x_j = jL/N$, whose Voronoi cells are the
  cubes of the grid, so the gas is the realisation itself, an exact Gaussian random field at grid resolution;
- **(b) uniform random points** (`uniform_generators`), a mesh unrelated to the field;
- **(c) density-weighted points** (`density_weighted_generators`): the number density of points is proportional to
  $1 + \delta$, so that the cells hold equal mass on average and are smaller where the gas is denser, as in a moving
  mesh.

In (b) and (c) every cell takes the realisation interpolated trilinearly at its generating point, and its own volume
(`voronoi_volumes`). The same realisation, $\sigma_\delta = 0.2$, goes through the three; the figure shows one plane of
each, and the power spectrum of the density they define, read at the grid nodes (each node takes the value of the cell
that holds it), with the estimator of §1.

In [ ]:
from skewerjax.simulations import grf_cells, uniform_generators, density_weighted_generators
L4, N4 = 50.0, 64
g4 = PowerLawGRF(A_s=0.04, k_0=2.0, n_s=-1.25, L=(L4,) * 3, N=(N4,) * 3)                    # sigma_delta = 0.2
delta4 = np.asarray(g4.sample(jax.random.PRNGKey(0)))
n_bar4 = {"HI": 1e-12}                                                                       # [comoving cm^-3]
M4 = N4 ** 3                                                                                 # as many generating points as grid cells
generators = {"(a) grid cells": None,
              "(b) uniform random points": uniform_generators(g4, M4, seed=1),
              "(c) density-weighted points": density_weighted_generators(delta4, g4, M4, seed=2)}
cells4 = {name: grf_cells(delta4, g4, n_bar4, generators=gen, temperature=1.5e4) for name, gen in generators.items()}
print(f"sigma_delta = {float(jnp.sqrt(g4.variance())):.2f}, min(1 + delta) = {1 + delta4.min():.3f}")
for name, c in cells4.items():
    print(f"{name:28s}: {c}; sizes V^(1/3) from {c.size_cMpc.min():.2f} to {c.size_cMpc.max():.2f} cMpc (median {np.median(c.size_cMpc):.2f})")

In [ ]:
from scipy.spatial import cKDTree
from skewerjax.core import grid_points
from skewerjax.utils.plotting import binned_power, tol_cmap
from skewerjax.utils import tutorial
import matplotlib.colors as mpl_colors
dx4, z4 = L4 / N4, (N4 // 2) * (L4 / N4)                                     # the plane of nodes z = L/2
nodes4 = grid_points(L4, g4.shape)
def at_nodes(c):                    # the density contrast of the cells at the grid nodes: each node takes the cell that holds it
    j = cKDTree(c.positions, boxsize=L4).query(nodes4, workers=-1)[1]
    return (c.species["HI"]["number_density"][j] / n_bar4["HI"] - 1.0).reshape(g4.shape)
zoom = 12.5                                                                  # a 12.5 cMpc corner of the plane
u = np.linspace(0.0, zoom, 500)
U, W = np.meshgrid(u, u, indexing="ij")
plane = np.stack([U.ravel(), W.ravel(), np.full(U.size, z4)], -1)
colors = dict(zip(cells4, (EC["HI"], TOL["red"], TOL["green"])))
fig = plt.figure(figsize=(24, 15), layout="constrained")
gs = fig.add_gridspec(2, 3)
for c_, (name, c) in enumerate(cells4.items()):
    ax = fig.add_subplot(gs[0, c_])
    owner = cKDTree(c.positions, boxsize=L4).query(plane, workers=-1)[1].reshape(U.shape)          # the cell of every pixel
    edge = (np.diff(owner, axis=0, prepend=owner[:1]) != 0) | (np.diff(owner, axis=1, prepend=owner[:, :1]) != 0)
    im = ax.imshow(np.log10(c.species["HI"]["number_density"][owner]).T, origin="lower", extent=(0, zoom, 0, zoom), cmap=tol_cmap("iridescent"),
                   vmin=-12.3, vmax=-11.7, interpolation="nearest")
    ax.imshow(np.ma.masked_where(~edge.T, edge.T), origin="lower", extent=(0, zoom, 0, zoom), cmap=mpl_colors.ListedColormap([FG]),
              alpha=0.35, interpolation="nearest")
    near = (np.abs(c.positions[:, 2] - z4) < 0.5 * dx4) & (c.positions[:, 0] < zoom) & (c.positions[:, 1] < zoom)
    ax.scatter(c.positions[near, 0], c.positions[near, 1], s=7, color=FG, lw=0)
    ax.set(xlabel="x [cMpc]", ylabel="y [cMpc]", title=name, aspect="equal")
fig.colorbar(im, ax=fig.axes[:3], label=r"$\log_{10} n_{\rm HI}$ [comoving cm$^{-3}$]", shrink=0.85)
edges4 = np.geomspace(0.999 * 2 * np.pi / L4, np.pi * N4 / L4, 19)
P_in4 = tutorial.binned_input_power(g4, edges4)                              # the input, averaged over the modes of each bin
meas = {name: binned_power(at_nodes(c)[None], L4, edges4) for name, c in cells4.items()}
axP, axR = fig.add_subplot(gs[1, :2]), fig.add_subplot(gs[1, 2])
kb4 = meas["(a) grid cells"][0]
axP.plot(kb4, P_in4, color=FG, lw=2.4, label="input, averaged over the modes of each bin")
for name, (kb, Pb, eb, _) in meas.items():
    axP.plot(kb, Pb, "o-" if name.startswith("(a)") else "s-", ms=6, lw=1.4, color=colors[name], label=name)
axP.axvline(np.pi / dx4, color=DIM, ls="--", lw=1.2)
axP.set(xscale="log", yscale="log", xlabel=r"$|k|$ [cMpc$^{-1}$]", ylabel=r"$P(k)$ [cMpc$^3$]", title="P(|k|) of the density of the cells, at the grid nodes")
axP.legend()
_, Pa, ea, _ = meas["(a) grid cells"]
ea = P_in4 * ea / Pa                                                         # mode counting about the input: P_in sqrt(2 / n_modes)
axR.errorbar(kb4, Pa / P_in4, ea / P_in4, fmt="o", ms=6, capsize=3, color=colors["(a) grid cells"], label="(a) / input")
for name in list(cells4)[1:]:
    axR.plot(kb4, meas[name][1] / Pa, "s-", ms=6, color=colors[name], label=f"{name[:3]} / (a), the same realisation")
axR.axhline(1, color=FG, lw=1.2); axR.axvline(np.pi / dx4, color=DIM, ls="--", lw=1.2)
axR.set(xscale="log", ylim=(0, 1.6), xlabel=r"$|k|$ [cMpc$^{-1}$]", ylabel="ratio", title="relative to the input"); axR.legend(fontsize=15)
add_panel_letters(fig.axes[:3] + [axP, axR])
fig.suptitle("One realisation, three sets of cells: the grid keeps the input spectrum, simulation-like generating points smooth it", fontsize=21)
plt.show()
for name in list(cells4)[1:]:
    r = meas[name][1] / Pa
    print(f"{name:28s}: P / P_(a) = {np.interp(0.3, kb4, r):.3f} at k = 0.3, {r[-1]:.3f} at the grid Nyquist pi / Delta = {np.pi / dx4:.2f} cMpc^-1")
print(f"(a) / input: chi2/dof = {np.mean(((Pa - P_in4) / ea) ** 2):.2f} over {kb4.size} bins (mode-counting errors about the input)")

*Reading the figure.*

- **(a)** The grid cells are the squares of the grid: the gas is the realisation itself, and its spectrum is the
  input within the scatter of one realisation (mode counting).
- **(b)** Uniform random points cut the same field into irregular cells. Every cell takes the field interpolated at its
  generating point and holds it over its own volume, which smooths the field twice: power is lost progressively
  above $k \sim 0.5\ \mathrm{cMpc^{-1}}$, down to about a third at the grid's Nyquist frequency (dashed), while the
  largest scales are untouched.
- **(c)** Density-weighted points put more, smaller cells where the gas is dense. With $\sigma_\delta = 0.2$ the cell
  volumes change by only $\sim 20\,\%$, so the spectrum is almost that of (b): what matters at this contrast is the
  irregular sampling, not where the points are. Simulation meshes follow far larger contrasts (notebook 05), where the
  resolution of the dense gas improves and that of the voids degrades.

**This is the object that notebook 05 produces from a simulation snapshot** (`snapshot_cells`): Voronoi cells with,
per species, a comoving density, a temperature and a peculiar velocity. From here on nothing depends on where the
cells came from: notebook 07 selects the cells along the skewers, cuts the skewers into the pieces of the cells and
sums their profiles into $\tau$.